In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### ResNet-18 skeleton (with the Dropout head used in training)

In [11]:
# 1. Update ResNet skeleton to match your trained weights
def get_resnet18(num_classes=8):
  model = models.resnet18(weights=None)
  num_ftrs = model.fc.in_features
  model.fc = nn.Linear(num_ftrs, num_classes)  # Direct Linear layer
  return model

### GoogLeNet skeleton (with the fc2 auxiliary heads)

In [12]:
class GoogLeNetTransfer(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.googlenet = models.googlenet(weights=None, aux_logits=True)
        num_ftrs = self.googlenet.fc.in_features
        self.googlenet.fc = nn.Linear(num_ftrs, num_classes)
        if self.googlenet.aux1 is not None:
            self.googlenet.aux1.fc2 = nn.Linear(self.googlenet.aux1.fc2.in_features, num_classes)
        if self.googlenet.aux2 is not None:
            self.googlenet.aux2.fc2 = nn.Linear(self.googlenet.aux2.fc2.in_features, num_classes)

    def forward(self, x):
        out = self.googlenet(x)
        return out[0] if isinstance(out, (tuple, list)) else out

### Loading the Model

In [13]:
# ==========================================
# 2. Load Checkpoints (With Prefix Fix)
# ==========================================
resnet = get_resnet18(num_classes=8)
googlenet = GoogLeNetTransfer(num_classes=8)

resnet_weights = "./outputs/transfer_model_v4_phase2.pth"
googlenet_weights = "./outputs/transfer_model_v5_phase2.pth"

# 1. Load and clean ResNet weights
resnet_state = torch.load(resnet_weights, map_location=device)
resnet_state_cleaned = {
    k.replace("resnet.", "", 1): v for k, v in resnet_state.items()
}
resnet.load_state_dict(resnet_state_cleaned)

# 2. Load GoogLeNet weights
googlenet_state = torch.load(googlenet_weights, map_location=device)
googlenet.load_state_dict(googlenet_state)

# 3. Set to evaluation mode
resnet.to(device).eval()
googlenet.to(device).eval()

print("Both models loaded successfully into evaluation mode!")

Both models loaded successfully into evaluation mode!


### Voter

In [9]:
class SoftVotingEnsemble(nn.Module):
    def __init__(self, model_a, model_b, weight_a=0.5, weight_b=0.5):
        super().__init__()
        self.model_a = model_a
        self.model_b = model_b
        self.weight_a = weight_a
        self.weight_b = weight_b

    def forward(self, x):
        # 1. Get raw unnormalized scores (logits)
        logits_a = self.model_a(x)
        logits_b = self.model_b(x)

        # 2. Turn scores into percentages (probabilities that sum to 1.0)
        probs_a = F.softmax(logits_a, dim=1)
        probs_b = F.softmax(logits_b, dim=1)

        # 3. Compute weighted average of both models' confidence
        blended_probs = (self.weight_a * probs_a) + (self.weight_b * probs_b)
        return blended_probs

# Initialize with an even 50/50 split
ensemble = SoftVotingEnsemble(resnet, googlenet, weight_a=0.5, weight_b=0.5).to(device)

### Loading the Data

In [14]:
from datacleaner_v2 import build_training_dataloaders

train_loader, val_loader = build_training_dataloaders(
    csv_path='./raw_data/train_labels_strId.csv', 
    img_dir='./raw_data/train/train', 
    batch_size=32
)

### Test Run

In [18]:
from torch._functorch._aot_autograd import aot_autograd_result
correct_res = 0
correct_goog = 0
correct_ens = 0
total_samples = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)

        out_res = resnet(images)
        out_goog = googlenet(images)

        # 2. Convert to probabilities
        probs_res = F.softmax(out_res, dim=1)
        probs_goog = F.softmax(out_goog, dim=1)

        # 3. Soft-vote blend (50/50)
        probs_ens = 0.5 * probs_res + 0.5 * probs_goog

        # 4. Extract class predictions
        pred_res = torch.max(probs_res, dim=1)[1]
        pred_goog = torch.max(probs_goog, dim=1)[1]
        pred_ens = torch.max(probs_ens, dim=1)[1]

        # Track correct counts
        correct_res += (pred_res == labels).sum().item()
        correct_goog += (pred_goog == labels).sum().item()
        correct_ens += (pred_ens == labels).sum().item()
        total_samples += labels.size(0)

print(f"Validation Samples Tested: {total_samples}")
print(f"ResNet-18 Accuracy:  {(correct_res / total_samples) * 100:.2f}%")
print(f"GoogLeNet Accuracy:  {(correct_goog / total_samples) * 100:.2f}%")
print(f"Ensemble Accuracy:   {(correct_ens / total_samples) * 100:.2f}%")

Validation Samples Tested: 12732
ResNet-18 Accuracy:  99.47%
GoogLeNet Accuracy:  97.86%
Ensemble Accuracy:   99.48%


### Loading Test Data

In [19]:
# 3. Define our mapping dictionary (Numbers back to Strings)
idx_to_class = {
    0: 'CatEye', 1: 'CatFace', 2: 'CatMouth', 3: 'CatNose',
    4: 'DogEye', 5: 'DogFace', 6: 'DogMouth', 7: 'DogNose'
}

In [24]:
from datacleaner_v2 import build_test_dataloader

test_loader, sub_df = build_test_dataloader(
    sample_sub_path='./sample_submission.csv', 
    img_dir='./raw_data/test/test', 
    batch_size=32
)

Test Pipeline Ready! Images to predict: 29443


### Testing on the Test Data

In [25]:
import torch
import torch.nn.functional as F

print("Generating final ensemble predictions...")
resnet.eval()
googlenet.eval()
predictions = []

with torch.no_grad():
  for images, ids in test_loader:
    images = images.to(device)

    # 1. Get class probabilities from both models
    probs_res = F.softmax(resnet(images), dim=1)
    probs_goog = F.softmax(googlenet(images), dim=1)

    # 2. Average the probabilities (50/50 soft voting)
    blended_probs = 0.5 * probs_res + 0.5 * probs_goog

    # 3. Get predicted class index and map to label name
    predicted_idx = torch.max(blended_probs, 1)[1]
    for idx in predicted_idx.cpu().numpy():
      predictions.append(idx_to_class[idx])

# 4. Save to CSV
sub_df['class'] = predictions
sub_df.to_csv('./outputs/my_eight_submission.csv', index=False)
print("Ensemble submission saved successfully to ./outputs/my_eight_submission.csv!")

Generating final ensemble predictions...
Ensemble submission saved successfully to ./outputs/my_eight_submission.csv!
